# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring** — ranking mature content for editorial refresh review.

**Status (2026-08-16): complete.** Every section is filled and reproducible, and this notebook runs top
to bottom in about a minute. It does not copy its model numbers from anywhere — it **refits the shipped
ranking inline and asserts the result matches the committed ML-08 receipt**, so a stale artifact fails the
run instead of quietly flattering the paper.

The headline: a hybrid ranking (the ML-07 rule picks the band and the reason codes, a logistic model orders
within it) places **45 of its top 50** pages on items measured as declining — precision@50 = **0.900**
against a **0.542** base rate, out-of-fold on clients the model never saw. The three results that argue
against that headline are in Sections 4 and 5, not omitted from them.

Written work lives in `work/capstone_report.md`; the assignment notebooks it draws on are
`w01_research_question.ipynb` (ML-02), `w02_ml_task_framing.ipynb` (ML-03), `w03_data_contract.ipynb`
(ML-04), `w03_feature_leakage_check.ipynb` (ML-05), `w04_signal_audit.ipynb` (ML-06),
`w04_baseline_score.ipynb` (ML-07), `w05_model.ipynb` (ML-08), `w06_validation_audit.ipynb` (ML-09) and
`w07_action_playbook.ipynb` (ML-10).

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is computed from the 30-day impression pair.
# So trend_direction, trend_pct, impressions_last_30d and impressions_prev_30d are never features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")


Working dir: C:\Users\real time\Desktop\Rayan_flyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421


## 1. Question

**Among mature indexed content items with established search demand, which specific pages are
undergoing measured organic decline and should be prioritised for editorial review in the coming
sprint?**

**The decision it supports.** A FlyRank content strategist has roughly 20–50 review slots per sprint
against a portfolio of tens of thousands of published pages. The binding constraint is editorial
capacity, so the useful output is an *ordering*, not a verdict on every page.

- **Unit of analysis:** one pseudonymized content item (`content_id`).
- **Output:** a priority score per page, delivered as a ranked queue with reason codes.
- **Action:** the editor opens the top K pages and either refreshes (update facts, realign headers with
  current intent, expand thin sections, refresh metadata) or, having looked, deliberately skips.
- **Cost of a wrong call:** a false positive burns a review slot worth roughly $300–$1,000 of editorial
  time on a stable page; a false negative leaves a decaying revenue page unreviewed while competitors
  take the position. The costs are asymmetric and capacity is fixed, which is why **precision at the top
  of the ranking** is the metric and accuracy is not.

**Task type:** ranking / scoring. A classifier is fitted underneath, but its probability is used as a
ranking score and evaluated with precision@K — never thresholded at 0.5 and reported as accuracy.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Section 1: the decision this output feeds is capacity-limited - that is what picks the metric.
CAPACITY_PER_SPRINT = 50
print(f"portfolio in this slice      {len(df):,} pages")
print(f"review slots per sprint      {CAPACITY_PER_SPRINT}")
print(f"share of portfolio acted on  {CAPACITY_PER_SPRINT / len(df):.2%}")
print(f"label base rate              {BASE_RATE:.4f}  <- every precision number sits next to this")

portfolio in this slice      30,000 pages
review slots per sprint      50
share of portfolio acted on  0.17%
label base rate              0.5421  <- every precision number sits next to this


## 2. Data

**Release used:** the anonymized starter dataset that ships in this repo —
`data/raw/content_refresh_anonymized.csv`, **30,000 rows × 44 columns**, one row per pseudonymized
content item, **32 pseudonymized clients**, trailing-90-day metrics. The gated warehouse release
(`hf://datasets/FlyRank/internship-warehouse`) was **not** used; where it would change a conclusion, that
is stated in Limitations.

**Date window.** The file contains **no date column at all** — verified, not assumed. Every time field is
a relative offset from an unstated snapshot date. Three window facts that follow:

- The two 30-day columns cover days 1–30 and 31–60; days 61–90 sit in the 90-day totals and in neither.
  `last_30d + prev_30d == impressions_90d` holds for only **8.7%** of rows.
- `days_with_impressions` caps at **88** while `days_with_sessions` reaches **90** — the GSC reporting lag,
  visible in the data. Search and analytics windows are not the same length.
- `content_age_days` has a minimum of exactly **90**: the file was pre-filtered upstream to mature pages,
  so the 30,000-row count is the file as shipped, not the product of my own filter.

**Excluded, and why.**

| Excluded | Why |
|---|---|
| `trend_direction`, `trend_pct` | The label source (`is_declining_label = trend_direction == "down"`). |
| `impressions_last_30d`, `impressions_prev_30d` | **Measured to reconstruct the label exactly (1.0000).** The label is a deterministic function of this pair, so they are leakage — a stronger exclusion than the docs' two named fields. |
| `clicks_last_30d/prev_30d`, `sessions_last_30d/prev_30d` | Same last-vs-prev ratio shape. Measured agreement with the label is only 0.536 / 0.538, so not leaks — dropped anyway as weak and hard to defend. |
| `provider_used`, `model_used` | Generation-provenance / product-decision flags (71.5% and 19.1% missing). They describe which internal pipeline wrote the page, not whether it is decaying. |
| `content_id`, `client_id` | Pseudonymous IDs — context only: grouping, joining, splitting, audit. Never features. |

That leaves **32 feature fields**. `w03_data_contract.ipynb` asserts in code that all 44 file columns plus
the engineered label land in exactly one bucket, so the contract cannot drift from the file.

**Public-safety.** No client names, domains, URLs or raw search queries appear anywhere in `work/`. IDs
are pseudonyms. Printouts in these notebooks show aggregate counts, and the top-20 review withholds
identifiers. The ranked queue CSV stays gitignored (`work/**/*.csv`); only metrics JSONs are committed.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Section 2: the data-safety claims above, re-checked here so the paper's numbers are self-verifying.
LABEL_DERIVED = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
CONTEXT = ["content_id", "client_id"]
EXCLUDED = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
            "provider_used", "model_used"]
FEATURES = [c for c in df.columns if c not in LABEL_DERIVED + CONTEXT + EXCLUDED + ["is_declining_label"]]

print(f"file columns {df.shape[1] - 1} + engineered label | features {len(FEATURES)} | "
      f"label-derived {len(LABEL_DERIVED)} | context {len(CONTEXT)} | excluded {len(EXCLUDED)}")
assert not set(FEATURES) & set(LABEL_DERIVED + CONTEXT + EXCLUDED), "bucket overlap"

# The leakage measurement that justifies the strongest exclusion.
def reconstruct(last, prev):
    pct = np.where(prev > 0, (last - prev) / prev.where(prev > 0) * 100.0, np.nan)
    return np.where(prev == 0, 0, np.where(pct < -20.0, 1, 0))

print()
for metric in ("impressions", "clicks", "sessions"):
    agree = (reconstruct(df[f"{metric}_last_30d"], df[f"{metric}_prev_30d"]) == y).mean()
    verdict = "LEAK - excluded" if agree > 0.99 else "near base rate - not a leak"
    print(f"  {metric + ' 30d pair':22s} agreement {agree:.4f}  {verdict}")

print()
print("Window facts:")
tiles = (df["impressions_last_30d"] + df["impressions_prev_30d"] == df["impressions_90d"]).mean()
print(f"  30d columns tile the 90d window in {tiles:.1%} of rows")
print(f"  days_with_impressions max {df['days_with_impressions'].max()} (GSC) vs "
      f"days_with_sessions max {df['days_with_sessions'].max()} (GA4)")
print(f"  content_age_days min {df['content_age_days'].min()} -> file pre-filtered to mature pages")
print()
print("Public-safety scan of the columns this project reads:")
UNSAFE_HINTS = ("url", "domain", "title", "query", "keyword_text", "client_name", "slug")
hits = [c for c in df.columns if any(h in c.lower() for h in UNSAFE_HINTS)]
print(f"  columns that could carry identifying text: {hits or 'none'}")

file columns 44 + engineered label | features 32 | label-derived 4 | context 2 | excluded 6

  impressions 30d pair   agreement 1.0000  LEAK - excluded
  clicks 30d pair        agreement 0.5364  near base rate - not a leak
  sessions 30d pair      agreement 0.5383  near base rate - not a leak

Window facts:
  30d columns tile the 90d window in 8.7% of rows
  days_with_impressions max 88 (GSC) vs days_with_sessions max 90 (GA4)
  content_age_days min 90 -> file pre-filtered to mature pages

Public-safety scan of the columns this project reads:
  columns that could carry identifying text: none


## 3. Methodology

**Assumptions, stated so they can be attacked.**

1. The dataset's `trend_direction` is an acceptable *proxy* for "in measured decline". It is not an
   observed editorial outcome and not a recovery measurement.
2. A page's trailing-90-day performance shape carries information about whether it is decaying.
3. Editorial capacity is fixed and small, so the top of the ranking is the only part that matters.

**Label.** `is_declining_label = 1` when `trend_direction == "down"`, i.e. impressions fell more than 20%
between the most recent 30 days and the 30 before that. Base rate **0.5421** (16,262 of 30,000).

Two properties of this label that shape every claim downstream:

- It is **defined, not observed** — a threshold on a ratio, so the model learns the dataset's definition
  of decline rather than the world's.
- It is **noise-sensitive at low volume**: 19.1% of declining pages have fewer than 100 impressions in
  90 days, where 5 → 3 impressions reads as "down 40%". And it **cannot fire at all** for the 3,388 rows
  with `impressions_prev_30d == 0`.

**Features.** The 32 contract-approved fields: demand/market (`search_volume`, `competition`,
`competition_level`, `cpc`), page shape (`word_count`, `char_count`, tiers, `content_type`, `main_intent`),
90-day totals (impressions, clicks, pageviews, sessions, users, engaged sessions, AI sessions, scroll
events), coverage (`days_with_impressions`, `days_with_sessions`), age/freshness (`content_age_days`,
`age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`), and rates/position (`ctr`,
`avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`).

Handling that is not optional here: **`avg_position == 0` means "no data"** (1,205 rows) so it becomes
missing plus a flag, never a zero; and **missingness follows `content_type`** — `feedly article` is 100%
missing `search_volume`/`competition`/`cpc`, `keyword article` 28.3% missing `word_count` — while the label
rate differs sharply by type (28.7% vs 56.1% vs 57.2%). A blind `fillna(0)` would hand the model a proxy
for "this is a feedly article" disguised as a demand feature, so missing values get explicit `has_*`
indicator flags.

**Baseline (built, ML-07).** A transparent five-condition additive score, no fitted weights:
`3×established_coverage + 2×has_demand + 2×mid_position + 1×stale_90d + 1×mature_page`, with reason codes
on every row. Disclosed weakness: its thresholds were read off the ML-03 label-rate crosstabs, so it is
**mildly tuned in-sample** and its numbers here are optimistic.

**Validation design.** **Grouped by `client_id`** — no client may appear in both train and test. Per-client
label rates span **0.000 to 0.937** across 32 clients, and the three largest clients hold **43.3%** of all
rows, so a random row split would measure client memorisation rather than generalisation. A **time-aware**
split is impossible from this file (no date column); that needs the warehouse release. Seed 42 throughout.

**Leakage checks run.** (a) The label-reconstruction test above, which produced the exclusion of the
30-day impression pair. (b) A code assertion that the baseline's five inputs contain no label-derived or
ID column. (c) An AUC sanity check — the baseline's ROC-AUC is **0.6485**, and a near-1.0 value from five
hand-written conditions would have been the alarm. (d) The contract's bucket-overlap assertions.

**Model and validation (built, ML-08/ML-09).** Four candidates - logistic regression, a depth-3 tree, a
random forest and histogram gradient boosting - each used as a ranking score and compared against this
baseline on the same client-grouped split, with permutation importance read on the winner. The shipped
system is a **hybrid**: this rule selects the band and supplies the reason codes, a logistic model orders
pages within it. Section 4 has the table.

**A fifth leakage check, added by ML-09 and worth stating here because it changes how the others should be
read.** Injecting the two forbidden columns back moves out-of-fold AUC from 0.677 to **0.920 as raw counts**
but to **0.9996 in log space** - the same columns, the same model. The label is a threshold on a *ratio*,
which is linear in logs and not in raw counts, so **leakage is a property of (column, representation,
model), not of a column**. A screen run before the transform decision can under-state a leak by 0.08 AUC.

In [4]:
# Rebuild the ML-07 baseline inline, so this notebook stands alone (no dependency on
# another notebook having been run first). Then cross-check against the committed metrics file.
import json
from pathlib import Path

position = df["avg_position"].replace(0, np.nan)  # 0 means "no data", not rank 0
RULE = {
    "established_coverage": (df["days_with_impressions"].between(20, 87), 3),
    "has_demand":           (df["impressions_90d"] >= 40, 2),
    "mid_position":         (((position > 3) & (position <= 50)).fillna(False), 2),
    "stale_90d":            (df["days_since_last_update"] >= 90, 1),
    "mature_page":          (df["content_age_days"].between(90, 364), 1),
}
baseline = df[["content_id", "client_id", "is_declining_label", "impressions_90d",
               "days_with_impressions", "avg_position", "ctr", "days_since_last_update",
               "content_age_days", "content_type"]].copy()
baseline["baseline_score"] = sum(flag.astype(int) * pts for flag, pts in RULE.values())
flag_frame = pd.DataFrame({name: flag.to_numpy() for name, (flag, _) in RULE.items()})
baseline["reason_codes"] = [",".join(flag_frame.columns[row]) or "no_signal" for row in flag_frame.to_numpy()]
baseline["rank_key"] = baseline["baseline_score"] * 100 + np.minimum(np.log1p(baseline["impressions_90d"]), 10)

queue = baseline.sort_values("rank_key", ascending=False, kind="stable").reset_index(drop=True)
queue.insert(0, "queue_rank", np.arange(1, len(queue) + 1))
y_queue = queue["is_declining_label"].to_numpy()

K_VALUES = [20, 50, 100, 200, 500]
baseline_metrics = {f"precision_at_{k}": round(float(y_queue[:k].mean()), 4) for k in K_VALUES}
print(f"base rate {BASE_RATE:.4f}")
for k in K_VALUES:
    print(f"  baseline precision@{k:<4d} {baseline_metrics[f'precision_at_{k}']:.4f}")

# Reproducibility check: do these match the receipt committed by w04_baseline_score.ipynb?
receipt = Path("work/outputs/baseline_metrics.json")
if receipt.exists():
    committed = json.loads(receipt.read_text())
    same = all(committed[k] == v for k, v in baseline_metrics.items())
    print(f"\nmatches work/outputs/baseline_metrics.json: {same}")
    assert same, "this notebook and the committed receipt disagree - one of them is stale"
else:
    print("\nwork/outputs/baseline_metrics.json not found - run w04_baseline_score.ipynb to write it")


base rate 0.5421
  baseline precision@20   0.7500
  baseline precision@50   0.7400
  baseline precision@100  0.8000
  baseline precision@200  0.8350
  baseline precision@500  0.8080

matches work/outputs/baseline_metrics.json: True


## 4. Results (vs baseline)

**The honest table.** Every model row is **out-of-fold on a client-held-out split** (GroupKFold(5) on
`client_id`, zero overlap asserted); the baseline is the unchanged ML-07 rule scored on identical rows.
Full working: `w05_model.ipynb`; receipt: `work/outputs/model_metrics.json`.

| System | Split | p@20 | p@50 | p@100 | p@200 | ROC-AUC | Base rate |
|---|---|---|---|---|---|---|---|
| Flag everything (floor) | — | 0.542 | 0.542 | 0.542 | 0.542 | 0.500 | 0.542 |
| My rule baseline (ML-07) | full data, in-sample | 0.750 | 0.740 | 0.800 | **0.835** | 0.649 | 0.542 |
| Decision tree (depth 3) | client-held-out | 0.600 | 0.600 | 0.570 | 0.600 | 0.624 | 0.542 |
| Random forest | client-held-out | 0.550 | 0.680 | 0.740 | 0.780 | 0.681 | 0.542 |
| Histogram gradient boosting | client-held-out | 0.900 | 0.820 | 0.790 | 0.800 | **0.691** | 0.542 |
| Logistic regression | client-held-out | 0.800 | 0.880 | 0.810 | 0.805 | 0.677 | 0.542 |
| **Hybrid — rule band + logistic order (shipped)** | client-held-out | **0.950** | **0.900** | **0.850** | 0.805 | 0.661 | 0.542 |
| *Reference pipeline, random forest* | *client holdout* | *—* | *0.740* | *—* | *—* | *0.750* | *0.542* |
| *Reference pipeline, rule baseline* | *full data* | *—* | *0.240* | *—* | *—* | *0.627* | *0.542* |

The two italic rows are **the repo's bundled results** (`outputs/model_report.md`), not mine.

**What the shipped row means.** Of 50 review slots, **45** land on pages measured as declining, against
**37** for my rule and **27** for random triage. That is a 1.66× lift on the metric that matches the
decision — and unlike the baseline row, it is measured on clients the model never saw.

**The falsifiable prediction I recorded before training, and its result.** ML-07's defect was specific:
*good band, bad ordering inside it*, visible as precision that **rises** with K. The test was whether a
model makes the curve fall. Measured:

| K | 10 | 20 | 50 | 100 | 200 | 500 |
|---|---|---|---|---|---|---|
| ML-07 rule | 0.700 | 0.750 | 0.740 | 0.800 | 0.835 | 0.808 |
| Boosting | 1.000 | 0.900 | 0.820 | 0.790 | 0.800 | 0.812 |
| Logistic | 0.700 | 0.800 | 0.880 | 0.810 | 0.805 | 0.782 |
| **Hybrid** | **1.000** | **0.950** | **0.900** | **0.850** | 0.805 | 0.770 |

**Only the hybrid is monotone decreasing across every K measured** — asserted in code below, not eyeballed.
Plain logistic regression peaks at K=50 and boosting wobbles. The prediction was falsifiable, three of the
four candidates failed it, and one passed.

**Four caveats that belong in the same breath as the number:**

1. **The comparison is tilted toward the baseline, not the model.** The rule's thresholds were read off
   crosstabs over all 30,000 rows, so it is mildly in-sample everywhere while every model number is
   strictly out-of-fold. I did not re-fit the rule per fold, because the rule *as submitted for ML-07* is
   the thing that has to be beaten.
2. **precision@50 is 50 rows; one row moves it 0.02.** Bootstrapped: hybrid 0.900 **[0.820, 0.980]**, rule
   0.740 **[0.620, 0.860]**. Those overlap slightly. Measured and directional — not established.
3. **The per-fold picture picks a different winner.** Inside individual held-out folds, boosting leads on
   mean (0.828) and on the worst fold (0.780); the hybrid is second (0.820 / 0.740) and plain logistic
   collapses to 0.560 on one fold. Pooled ranking and per-fold ranking answer different questions and I
   report both rather than the flattering one.
4. **The rule still wins at K=200.** If capacity were 200 pages a sprint instead of 50, the recommendation
   would change. The metric is only right because the capacity is what it is.

**The result I did not expect.** I ordered the candidate list by readability so a simple winner would be
obvious if it happened — and the **logistic regression beat both tree ensembles**, but only after the
count columns were `log1p`'d. Without that step it scored 0.660 and finished last. The lesson is not that
linear models are underrated; it is that **the ML-05 feature-engineering decision mattered more than the
choice of model**, and skipping it "because trees do not need it" would have produced the opposite
conclusion and a wrong story about why.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Section 4: the results table. The model row is REPRODUCED here, not copied - this notebook refits the
# shipped system (rule band + logistic) inline in ~30s and asserts it matches the ML-08 receipt.
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from pandas.api.types import is_numeric_dtype

model_metrics = json.loads(Path("work/outputs/model_metrics.json").read_text())

def roc_auc(labels, scores) -> float:
    """AUC via the rank-sum identity - no sklearn needed."""
    labels = np.asarray(labels)
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(np.asarray(scores, dtype=float)).rank().to_numpy()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def precision_at_k(labels, scores, k: int) -> float:
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

baseline_auc = roc_auc(y_queue, queue["baseline_score"])

# --- refit the shipped system on the client-held-out split -----------------
FEATURES = [
    "search_volume", "competition", "competition_level", "cpc",
    "word_count", "char_count", "word_count_tier", "char_count_tier",
    "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier", "age_tier_order", "days_since_last_update", "freshness_tier",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
]
X = df[FEATURES].copy()
X["avg_position"] = X["avg_position"].replace(0, np.nan)
for c in ["search_volume", "competition", "cpc", "word_count", "char_count", "avg_position"]:
    X[f"has_{c}"] = X[c].notna().astype(int)
COUNT_COLS = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
              "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]
X[COUNT_COLS] = np.log1p(X[COUNT_COLS])
CATEGORICAL = [c for c in FEATURES if not is_numeric_dtype(df[c])]
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]

FORBIDDEN = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "is_declining_label", "content_id", "client_id"}
assert not FORBIDDEN & set(X.columns), "a forbidden column reached the model"

pre = ColumnTransformer([
    ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="__missing__")),
                      ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), CATEGORICAL),
])
LOGIT = LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)

groups = df["client_id"].to_numpy()
oof = np.full(len(df), np.nan)
for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
    assert not (set(groups[tr]) & set(groups[te])), "client leaked across the split"
    pipe = Pipeline([("pre", clone(pre)), ("clf", clone(LOGIT))]).fit(X.iloc[tr], y[tr])
    oof[te] = pipe.predict_proba(X.iloc[te])[:, 1]
assert not np.isnan(oof).any(), "a row was never scored out-of-fold"

# `baseline` is df-aligned (built from df without sorting), so its columns line up with oof.
band = baseline["baseline_score"].to_numpy()
BASELINE_KEY = baseline["rank_key"].to_numpy()
hybrid = band * 1000 + oof * 100          # rule band first, model orders inside it

# --- the table -------------------------------------------------------------
KS = [20, 50, 100, 200]
rows = [
    {"system": "flag everything (floor)", "split": "-",
     **{f"p@{k}": round(BASE_RATE, 3) for k in KS}, "roc_auc": 0.500},
    {"system": "my rule baseline (ML-07)", "split": "full data, in-sample",
     **{f"p@{k}": baseline_metrics[f"precision_at_{k}"] for k in KS}, "roc_auc": round(baseline_auc, 4)},
    {"system": "logistic (ML-08)", "split": "client-held-out",
     **{f"p@{k}": round(precision_at_k(y, oof, k), 3) for k in KS}, "roc_auc": round(roc_auc(y, oof), 4)},
    {"system": "HYBRID - shipped (ML-08)", "split": "client-held-out",
     **{f"p@{k}": round(precision_at_k(y, hybrid, k), 3) for k in KS}, "roc_auc": round(roc_auc(y, hybrid), 4)},
]
print(pd.DataFrame(rows).set_index("system").to_string())
print(f"\nbase rate {BASE_RATE:.4f}")

# --- reproduced numbers must match the ML-08 receipt -----------------------
receipt = model_metrics["systems"]["hybrid_rule_band_plus_logistic"]
here = round(precision_at_k(y, hybrid, 50), 4)
print(f"\nhybrid p@50 reproduced here = {here} | ML-08 receipt = {receipt['p@50']}")
assert here == receipt["p@50"], "this notebook disagrees with the committed ML-08 receipt"
print("  -> matches. The paper's model numbers are reproducible from a fresh clone, not copied.")

print(f"\nslots on a genuinely declining page at K=50: "
      f"{precision_at_k(y, hybrid, 50)*50:.0f}/50 (hybrid) vs "
      f"{baseline_metrics['precision_at_50']*50:.0f}/50 (rule) vs {BASE_RATE*50:.0f}/50 (random)")
print(f"lift over the base rate: {precision_at_k(y, hybrid, 50)/BASE_RATE:.2f}x")

# --- the falsifiable prediction, asserted rather than eyeballed ------------
print("\nDid the model fix the ordering? (ML-07's defect: precision RISES with K)")
curves = {"ML-07 rule": BASELINE_KEY, "logistic": oof, "hybrid": hybrid}
print(f"  {'K':>5}" + "".join(f"{n:>14}" for n in curves))
for k in [10, 20, 50, 100, 200, 500]:
    print(f"  {k:>5}" + "".join(f"{precision_at_k(y, s, k):>14.3f}" for s in curves.values()))
for n, s in curves.items():
    vals = [precision_at_k(y, s, k) for k in [10, 20, 50, 100, 200, 500]]
    print(f"  {n:<12} monotone decreasing: {all(a >= b for a, b in zip(vals, vals[1:]))}")
assert all(a >= b for a, b in zip([precision_at_k(y, hybrid, k) for k in [10, 20, 50, 100, 200, 500]],
                                  [precision_at_k(y, hybrid, k) for k in [20, 50, 100, 200, 500]]))
print("  -> the shipped ranking passes the test recorded before training; the rule does not.")

print(f"\nbaseline ROC-AUC {baseline_auc:.4f} | hybrid ROC-AUC {roc_auc(y, hybrid):.4f}")
print("  note: the hybrid's AUC is LOWER than plain logistic (0.6774) - gating on the rule's band")
print("  discards ordering information below the band. I optimise the top of a queue, so I accept that.")
assert baseline_auc < 0.90, "suspiciously high AUC for a hand-written rule - check for leakage"

                                         split   p@20   p@50  p@100  p@200  roc_auc
system                                                                             
flag everything (floor)                      -  0.542  0.542  0.542  0.542   0.5000
my rule baseline (ML-07)  full data, in-sample  0.750  0.740  0.800  0.835   0.6485
logistic (ML-08)               client-held-out  0.800  0.880  0.810  0.805   0.6774
HYBRID - shipped (ML-08)       client-held-out  0.950  0.900  0.850  0.805   0.6606

base rate 0.5421

hybrid p@50 reproduced here = 0.9 | ML-08 receipt = 0.9
  -> matches. The paper's model numbers are reproducible from a fresh clone, not copied.

slots on a genuinely declining page at K=50: 45/50 (hybrid) vs 37/50 (rule) vs 27/50 (random)
lift over the base rate: 1.66x

Did the model fix the ordering? (ML-07's defect: precision RISES with K)
      K    ML-07 rule      logistic        hybrid
     10         0.700         0.700         1.000
     20         0.750         0.

  logistic     monotone decreasing: False
  hybrid       monotone decreasing: True
  -> the shipped ranking passes the test recorded before training; the rule does not.

baseline ROC-AUC 0.6485 | hybrid ROC-AUC 0.6606
  note: the hybrid's AUC is LOWER than plain logistic (0.6774) - gating on the rule's band
  discards ordering information below the band. I optimise the top of a queue, so I accept that.


## 5. Limitations

**What this work cannot claim.**

1. **No causal claim.** Nothing here records an intervention: no page was refreshed because of a score,
   and no outcome was measured afterwards. "Refreshing these pages will recover traffic" is unsupported
   and would need a controlled or matched design. The output is a **review queue**.
2. **No forecast.** With no date column and no future window, every number is concurrent. The honest
   sentence is "this page resembles the pages measured as declining", never "this page will decline".
3. **No claim about search-engine behaviour.** I did not model or reverse-engineer any ranking algorithm.
   I modelled observable performance metrics in one pseudonymized portfolio.
4. **No editorial-quality claim.** The score reads traffic shape, not prose quality, factual accuracy or
   strategic value. A page can be decaying because the topic died, and no rewrite fixes that.
5. **The label is a definition, not an outcome** — and a noisy one at low volume (19.1% of declining
   pages have <100 impressions in 90 days). It also cannot fire for 3,388 rows where prior-30-day
   impressions are zero, which is why 46.6% of `feedly article` rows are structurally unlabelable as
   declining.
6. **Survivorship, twice over.** Pages younger than 90 days and pages with zero traffic were removed
   before I received the file. Any "x% of pages are declining" statement means *of mature pages that
   still get some traffic*.
7. **The panel is unbalanced.** 32 clients, 3 to 7,008 pages each, per-client label rates 0.000–0.937.
   Portfolio-wide averages are dominated by a few large clients.
8. **The baseline's numbers are in-sample** (thresholds read off this data's crosstabs) and therefore
   optimistic — which tilts the ML-08 comparison *toward the baseline*, not toward my model.
9. **1,205 pages are partly blind** — `avg_position == 0` means no position reading, not rank 1.
10. **The shipped queue has worse portfolio coverage than the baseline it replaces.** Top 50: the rule
    spans 8 of 32 clients with one supplying 44%; the hybrid spans **7 clients with one supplying 58%**
    (29 of 50 slots). Better ranking, worse coverage. A per-client cap is required before use — a product
    decision the metric cannot see.

**Limits that only became visible once a model existed:**

11. **It ranks by exposure, not by decay.** Permutation importance puts `impressions_90d` first by 2.4×
    over the next feature, with `avg_position` the only non-volume signal near the top. I flagged this as
    a risk *before* training and it reproduced. The model substantially sorts pages by how much
    measurable search activity they carry.
12. **It cannot tell a healthy page-1 page from a decaying one.** All five of the shipped queue's top-50
    misses sit at `avg_position` 4.1–10.5, and four of them *grew* (+23.6% to +93.7%). ML-06 measured the
    same blind spot from the other side: `top_3` pages decline least of any position band (0.241).
13. **It cannot rank `comparison article` at all** — out-of-fold AUC **0.524** on 697 pages, i.e. chance.
    Those rows should be dropped from the queue rather than trusted.
14. **It works far better for some clients than others.** Per-client out-of-fold AUC runs 0.506–0.770
    (median 0.642) among clients with ≥200 pages. One client has no AUC at all — zero positives — so the
    metric is undefined there and is reported as undefined rather than filled in.
15. **Cross-validation is not a sealed test.** Every fold's data was visible to me while working, and I
    chose the shipped configuration after seeing fold results. **No sealed-evaluation claim is made
    anywhere in this report.** ML-09 ran the full attack checklist against this feature set; the one
    check it could not clear is below.
17. **The label's window sits inside the feature window, and no version of this dataset can fix that.**
    The label lives in days 1-60; several features are 90-day aggregates containing those days. Three
    things stop it being fatal - the overlap is of magnitude not identity (`impressions_90d` alone scores
    AUC 0.585, and a sum cannot express a direction), the task is framed as concurrent detection rather
    than forecasting, and a clean version needs day-61-90 features this file cannot produce. Disclosed
    rather than passed.
16. **A single random-split mistake would have inflated AUC by 0.152** (0.777 vs 0.624). That is how large
    the client-memorisation effect is on this data, and it is the most likely way a reader reproducing
    this work gets a number that looks better than mine.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Section 5: the limits, quantified rather than asserted.
declining = df[df["trend_direction"] == "down"]
print(f"noise-sensitive label: {(declining['impressions_90d'] < 100).mean():.1%} of declining pages "
      f"have <100 impressions in 90 days (n={(declining['impressions_90d'] < 100).sum():,})")
print(f"label cannot fire:     {(df['impressions_prev_30d'] == 0).sum():,} rows (prev-30d impressions == 0)")
print(f"partly blind:          {(df['avg_position'] == 0).sum():,} rows with avg_position == 0")
print(f"survivorship:          min content_age_days {df['content_age_days'].min()}, "
      f"rows with zero 90d impressions {(df['impressions_90d'] == 0).sum()}")

per_client = df.groupby("client_id").agg(pages=("content_id", "size"), rate=("is_declining_label", "mean"))
print(f"panel imbalance:       {len(per_client)} clients, {per_client['pages'].min()}-{per_client['pages'].max():,} "
      f"pages each, label rate {per_client['rate'].min():.3f}-{per_client['rate'].max():.3f}")
print(f"                       top 3 clients hold {per_client['pages'].nlargest(3).sum() / len(df):.1%} of rows")
print()

# --- limits that only exist because a model exists -------------------------
print("LIMIT 10 - the shipped queue covers the portfolio WORSE than the rule it replaces:")
for label, sc in [("ML-07 rule", BASELINE_KEY), ("hybrid (shipped)", hybrid)]:
    head = df.iloc[np.argsort(-np.asarray(sc, dtype=float), kind="stable")[:50]]
    vc = head["client_id"].value_counts()
    print(f"  {label:<18} top 50 spans {vc.size} of {len(per_client)} clients | "
          f"largest single client {vc.iloc[0]}/50 ({vc.iloc[0]/50:.0%})")

print("\nLIMIT 11 - it ranks by exposure, not decay (permutation importance, ML-08 receipt):")
for feat, drop in model_metrics["top_features_permutation_auc_drop"].items():
    print(f"  {feat:<24} AUC drop {drop:.4f}")

order = np.argsort(-hybrid, kind="stable")
top50 = df.iloc[order[:50]]
misses = top50[top50["is_declining_label"] == 0]
print(f"\nLIMIT 12 - every top-50 miss is a page-1 page ({len(misses)} of 50 misses):")
print(misses[["impressions_90d", "avg_position", "trend_direction", "trend_pct"]].round(2).to_string(index=False))
print(f"  avg_position range {misses['avg_position'].min():.1f}-{misses['avg_position'].max():.1f} "
      f"| measured 'up': {(misses['trend_direction'] == 'up').sum()} of {len(misses)}")

print("\nLIMIT 13/14 - where the ranking has no signal:")
for t, g in df.groupby("content_type"):
    idx = g.index.to_numpy()
    print(f"  {t:<20} n={len(idx):>6,}  out-of-fold AUC {roc_auc(y[idx], oof[idx]):.3f}")
pc = [roc_auc(y[g.index], oof[g.index]) for _, g in df.groupby("client_id") if len(g) >= 200]
pc_defined = [v for v in pc if v == v]
print(f"  per-client AUC (>=200 pages): {min(pc_defined):.3f} to {max(pc_defined):.3f}, "
      f"median {np.median(pc_defined):.3f} | undefined (no positives): {len(pc) - len(pc_defined)}")

print(f"\nLIMIT 16 - the split mistake that would have flattered me: "
      f"random-split AUC {model_metrics['split_inflation_check']['random_row_split_auc']} vs "
      f"grouped {model_metrics['split_inflation_check']['grouped_client_split_auc']} "
      f"(+{model_metrics['split_inflation_check']['random_row_split_auc'] - model_metrics['split_inflation_check']['grouped_client_split_auc']:.3f})")

noise-sensitive label: 19.1% of declining pages have <100 impressions in 90 days (n=3,110)
label cannot fire:     3,388 rows (prev-30d impressions == 0)
partly blind:          1,205 rows with avg_position == 0
survivorship:          min content_age_days 90, rows with zero 90d impressions 0
panel imbalance:       32 clients, 3-7,008 pages each, label rate 0.000-0.937
                       top 3 clients hold 43.3% of rows

LIMIT 10 - the shipped queue covers the portfolio WORSE than the rule it replaces:
  ML-07 rule         top 50 spans 8 of 32 clients | largest single client 22/50 (44%)
  hybrid (shipped)   top 50 spans 7 of 32 clients | largest single client 29/50 (58%)

LIMIT 11 - it ranks by exposure, not decay (permutation importance, ML-08 receipt):
  avg_position             AUC drop 0.0281
  clicks_90d               AUC drop 0.0503
  impressions_90d          AUC drop 0.1185
  sessions_90d             AUC drop 0.0327
  users_90d                AUC drop 0.0323

LIMIT 12 - every t

  comparison article   n=   697  out-of-fold AUC 0.524
  feedly article       n= 2,096  out-of-fold AUC 0.843
  keyword article      n=27,207  out-of-fold AUC 0.660
  per-client AUC (>=200 pages): 0.506 to 0.770, median 0.642 | undefined (no positives): 1

LIMIT 16 - the split mistake that would have flattered me: random-split AUC 0.7765 vs grouped 0.6244 (+0.152)


## 6. Ranked recommendations

**What a FlyRank editor would do tomorrow, from the shipped queue.** The ranking is the ML-08 hybrid: the
ML-07 rule selects the band and supplies the reason codes, the logistic model orders pages within it.
Every row is a **review** recommendation — the system never recommends publishing a change, only opening
a page.

**All 50 rows of the shipped queue sit at baseline score 9**, so all five conditions fired on every one of
them. Each recommendation still arrives with the same auditable explanation an editor could read in ML-07;
the model changed only the order, which is precisely the part the rule was measured to get wrong.

| Reason-code pattern | Recommended action | Confidence | Honest caveat |
|---|---|---|---|
| All five codes fired, model-ranked top 50 | **Refresh review** — check intent alignment, update facts, expand thin sections | Medium-high — 45 of 50 measured declining (0.900 vs a 0.542 base rate) | Off-season intent looks identical; no seasonality field exists to rule it out |
| All five codes, **page-1 position** (`avg_position` < 11) | **Refresh review, but confirm the direction first** | Medium — every measured error in the queue sits here | 4 of the 5 top-50 misses are page-1 pages that *grew* (+23.6% to +93.7%) |
| Any `comparison article` | **Drop from the queue** | None — measured out-of-fold AUC **0.524**, i.e. chance | 697 pages the model cannot order at all; the rule cannot either |
| `established_coverage` + `has_demand`, no `stale_90d` | **Monitor** | Low-medium | Recently updated; a second refresh is unlikely to be the lever |
| Missing `mid_position` (top-3 or no position data) | **Deprioritise** | Medium | `top_3` pages decline least (24.1%); `avg_position == 0` means no reading at all |
| `no_signal` | **Leave alone** | — | 75 pages, and their measured decline rate (0.293) is below the base rate |

**How to run the sprint, concretely:**

1. Take the top 50 rows of `work/outputs/model_action_score.csv`, ranked by `hybrid_score`.
2. **Confirm the direction on the page-1 rows before assigning them.** That is where every measured false
   positive sits, and strong position is ambiguous evidence in this data (ML-06: `top_3` declines least of
   any position band).
3. **Cap at ~8 pages per client.** This matters *more* for the shipped queue than for the rule it
   replaces: the hybrid's top 50 spans 7 of 32 clients with one supplying **58%** (29 of 50 slots), against
   the rule's 44%. Better ranking, worse coverage.
4. **Drop `comparison article` rows** until a model exists that can rank them.
5. Review the remainder by hand; the reason codes tell the editor what to look at first.
6. Log the decision (refreshed / skipped / monitored) **with a date**. That log is the observed outcome
   this project lacks, and it is what would make a real past→future label possible next quarter.

**Confidence, stated plainly.** Directional and decision-support. The queue is measurably better than
random triage at the metric that matches the decision (**0.900 vs 0.542 at K=50**, out-of-fold on clients
the model never saw), on one 90-day snapshot of 32 pseudonymized clients, with a bootstrap interval of
[0.820, 0.980] that only just clears the baseline's. It is **not** a sealed evaluation, **not** a forecast,
and it does **not** establish that refreshing anything recovers traffic.

**The one number to watch after the filters.** The operational rules below cost real precision — capping
per client and dropping the ambiguous rows removes pages the model ranked highly. That trade is a product
decision, not a better model, and the cell reports both numbers so nobody confuses the two.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Section 6: the actual recommendation output an editor would receive, from the SHIPPED hybrid ranking.
order = np.argsort(-hybrid, kind="stable")
sprint = df.iloc[order[:50]][["content_id", "client_id", "content_type", "impressions_90d",
                              "avg_position", "days_with_impressions", "is_declining_label"]].copy()
sprint["hybrid_score"] = hybrid[order[:50]]
sprint["reason_codes"] = baseline["reason_codes"].to_numpy()[order[:50]]
sprint["baseline_score"] = band[order[:50]]
sprint = sprint.reset_index(drop=True)
sprint.insert(0, "queue_rank", np.arange(1, len(sprint) + 1))

sprint["action"] = "refresh_review"

# Rule 2: page-1 rows need a direction check - every measured miss is one of these.
sprint.loc[sprint["avg_position"] < 11, "action"] = "refresh_review_verify_direction"

# Rule 4: the model cannot rank comparison articles at all (out-of-fold AUC 0.524).
sprint.loc[sprint["content_type"] == "comparison article", "action"] = "dropped_model_cannot_rank"

# Rule 3: cap per client so one portfolio cannot own the sprint.
PER_CLIENT_CAP = 8
sprint["rank_in_client"] = sprint.groupby("client_id").cumcount() + 1
sprint.loc[sprint["rank_in_client"] > PER_CLIENT_CAP, "action"] = "deferred_client_cap"

print("Sprint queue composition after the operational rules:")
print(sprint["action"].value_counts().to_string())
print()

sendable = sprint[sprint["action"].str.startswith("refresh_review")]
print(f"pages actually sent to an editor: {len(sendable)} of 50")
print(f"  of those, measured declining:   {sendable['is_declining_label'].mean():.3f} "
      f"(base rate {BASE_RATE:.3f})")
print(f"  distinct clients represented:   {sendable['client_id'].nunique()} "
      f"(unfiltered top 50: {sprint['client_id'].nunique()})")
print()
print("Reason-code patterns in the sendable set (counts only, no identifiers):")
print(sendable["reason_codes"].value_counts().head().to_string())
print()
print("The two numbers, kept apart on purpose:")
print(f"  raw model precision@50            {precision_at_k(y, hybrid, 50):.3f}  <- the honest headline")
print(f"  precision of the filtered sendable set {sendable['is_declining_label'].mean():.3f}  "
      f"<- a product decision layered on top, NOT a better model")
print("  The filters trade measured precision for portfolio coverage and safety. That is a choice a")
print("  human made, and it must never be reported as model performance.")

Sprint queue composition after the operational rules:
action
deferred_client_cap                23
refresh_review_verify_direction    22
refresh_review                      5

pages actually sent to an editor: 27 of 50
  of those, measured declining:   0.926 (base rate 0.542)
  distinct clients represented:   7 (unfiltered top 50: 7)

Reason-code patterns in the sendable set (counts only, no identifiers):
reason_codes
established_coverage,has_demand,mid_position,stale_90d,mature_page    27

The two numbers, kept apart on purpose:
  raw model precision@50            0.900  <- the honest headline
  precision of the filtered sendable set 0.926  <- a product decision layered on top, NOT a better model
  The filters trade measured precision for portfolio coverage and safety. That is a choice a
  human made, and it must never be reported as model performance.


## 7. Artifacts the paper embeds

Three figures, written to `work/figures/` as SVG, each with a table view underneath — because a figure
should never be the only way to read a number.

- **Figure 1 — `fig1_label_rate_by_score.svg`:** measured decline rate by baseline score level, with the
  base rate as a reference line. Shows both the signal (0.088 → 0.714) and the honest defect (the score is
  not monotone).
- **Figure 2 — `fig2_precision_at_k.svg`:** precision@K for the ML-07 rule *and* the shipped hybrid,
  against the base rate. **This is the figure that carries the paper's central claim**: the rule's curve
  climbs with K (mis-ordered at the top), the hybrid's falls from 1.000 (ordered correctly where the queue
  is actually read).
- **Figure 3 — `fig3_fold_spread.svg`:** precision@50 inside each of the five client-held-out folds, for
  every system. This is the figure that argues *against* my own choice — it shows boosting holding a higher
  floor than the hybrid I shipped, and plain logistic regression collapsing to 0.560 on one fold.

- **Figure 4 - `fig4_playbook_funnel.svg`** (written by ML-10, `w07_action_playbook.ipynb`): what the
  operational rules do to a sprint queue. It shows the rules removing *slots* rather than errors - the
  honest picture of a product decision, drawn so the raw and filtered numbers sit side by side and cannot
  be confused for one another.

Two series on a light surface, using slots 1 and 2 of the project's validated palette, distinguished by
colour **and** marker shape so the chart survives greyscale printing and colour-vision differences. The
base rate appears as a labelled reference line in Figures 2, 3 and 4, so no precision number is ever shown
without it.

In [8]:
# Three figures the paper embeds. Written to work/figures/ as SVG.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Colors: slots 1-2 of the validated reference palette, on a light chart surface.
SERIES_1 = "#2a78d6"
SERIES_2 = "#c8622d"
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
GRID = "#e3e2dd"

fig_dir = Path("work/figures")
fig_dir.mkdir(parents=True, exist_ok=True)


def style_axes(ax):
    """Recessive grid and axes; the data carries the chart."""
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=0)


# --- Figure 1: label rate by rule score level -----------------------------
by_score = baseline.groupby("baseline_score")["is_declining_label"].agg(["size", "mean"])

fig, ax = plt.subplots(figsize=(7.2, 4.0), facecolor=SURFACE)
style_axes(ax)
ax.bar(by_score.index, by_score["mean"], width=0.62, color=SERIES_1)
ax.axhline(BASE_RATE, color=INK_MUTED, linewidth=1.4, linestyle=(0, (5, 4)))
ax.annotate(f"base rate {BASE_RATE:.3f}\n(flag everything)",
            xy=(0.4, BASE_RATE), xytext=(0.4, BASE_RATE + 0.06),
            color=INK_MUTED, fontsize=8.5, va="bottom")
top = by_score["mean"].idxmax()
ax.annotate(f"{by_score.loc[top, 'mean']:.3f}  (n={int(by_score.loc[top, 'size']):,})",
            xy=(top, by_score.loc[top, "mean"]), xytext=(0, 6), textcoords="offset points",
            ha="center", color=INK, fontsize=9, fontweight="bold")
ax.set_title("Measured decline rate rises with the rule's score - but not monotonically",
             color=INK, fontsize=11.5, loc="left", pad=12)
ax.set_xlabel("Baseline rule score (0-9)", color=INK_MUTED, fontsize=9.5)
ax.set_ylabel("Share measured as declining", color=INK_MUTED, fontsize=9.5)
ax.set_xticks(by_score.index)
ax.set_ylim(0, 0.85)
fig.tight_layout()
fig.savefig(fig_dir / "fig1_label_rate_by_score.svg", format="svg", facecolor=SURFACE)
plt.close(fig)

# --- Figure 2: precision@K, rule vs shipped hybrid ------------------------
ks = [10] + K_VALUES
rule_ps = [precision_at_k(y, BASELINE_KEY, k) for k in ks]
hyb_ps = [precision_at_k(y, hybrid, k) for k in ks]

fig, ax = plt.subplots(figsize=(7.6, 4.4), facecolor=SURFACE)
style_axes(ax)
ax.plot(ks, hyb_ps, color=SERIES_1, linewidth=2.2, marker="o", markersize=8,
        markeredgecolor=SURFACE, markeredgewidth=2, label="Hybrid (shipped, out-of-fold)", zorder=3)
ax.plot(ks, rule_ps, color=SERIES_2, linewidth=2.2, marker="s", markersize=7,
        markeredgecolor=SURFACE, markeredgewidth=2, label="ML-07 rule (in-sample)", zorder=2)
ax.axhline(BASE_RATE, color=INK_MUTED, linewidth=1.4, linestyle=(0, (5, 4)))
ax.annotate(f"base rate {BASE_RATE:.3f}", xy=(ks[0], BASE_RATE), xytext=(2, -14),
            textcoords="offset points", color=INK_MUTED, fontsize=8.5)
ax.annotate(f"{hyb_ps[0]:.3f}", xy=(ks[0], hyb_ps[0]), xytext=(0, 9), textcoords="offset points",
            ha="center", color=INK, fontsize=9, fontweight="bold")
ax.annotate("the rule gets BETTER deeper in\nthe list - its top is mis-ordered",
            xy=(200, rule_ps[ks.index(200)]), xytext=(-8, -34), textcoords="offset points",
            ha="right", color=SERIES_2, fontsize=8.5)
ax.set_xscale("log")
ax.set_xticks(ks)
ax.set_xticklabels([str(k) for k in ks])
ax.set_title("The model fixes the ordering where the queue is actually read",
             color=INK, fontsize=11.5, loc="left", pad=12)
ax.set_xlabel("K (editorial capacity, pages reviewed)", color=INK_MUTED, fontsize=9.5)
ax.set_ylabel("Precision@K", color=INK_MUTED, fontsize=9.5)
ax.set_ylim(0.4, 1.05)
leg = ax.legend(frameon=False, fontsize=9, loc="lower left")
for t in leg.get_texts():
    t.set_color(INK_MUTED)
fig.tight_layout()
fig.savefig(fig_dir / "fig2_precision_at_k.svg", format="svg", facecolor=SURFACE)
plt.close(fig)

# --- Figure 3: the fold spread (the figure that argues against my choice) --
folds_pf = model_metrics["per_fold_precision_at_50"]
ORDER = ["decision_tree_d3", "logistic_regression", "random_forest",
         "baseline_rule", "hybrid", "hist_gradient_boost"]
LABELS = {"decision_tree_d3": "Decision tree (d3)", "logistic_regression": "Logistic regression",
          "random_forest": "Random forest", "baseline_rule": "ML-07 rule",
          "hybrid": "Hybrid (shipped)", "hist_gradient_boost": "Gradient boosting"}

fig, ax = plt.subplots(figsize=(7.6, 4.4), facecolor=SURFACE)
style_axes(ax)
ax.grid(axis="y", color=SURFACE)
ax.grid(axis="x", color=GRID, linewidth=0.8)
for i, name in enumerate(ORDER):
    vals = folds_pf[name]
    highlight = name in ("hybrid", "hist_gradient_boost")
    ax.plot(vals, [i] * len(vals), "o", markersize=7, color=SERIES_1 if highlight else GRID,
            markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3)
    ax.plot([min(vals), max(vals)], [i, i], color=SERIES_1 if highlight else GRID,
            linewidth=2.0, alpha=0.5, zorder=2)
    ax.plot(min(vals), i, "|", markersize=14, color=SERIES_2 if highlight else INK_MUTED, zorder=4)
ax.axvline(BASE_RATE, color=INK_MUTED, linewidth=1.4, linestyle=(0, (5, 4)))
ax.annotate(f"base rate {BASE_RATE:.3f}", xy=(BASE_RATE, len(ORDER) - 0.4), xytext=(4, 0),
            textcoords="offset points", color=INK_MUTED, fontsize=8.5)
ax.set_yticks(range(len(ORDER)))
ax.set_yticklabels([LABELS[n] for n in ORDER], fontsize=9)
ax.set_title("Each system's precision@50 across the 5 client-held-out folds (bar = worst fold)",
             color=INK, fontsize=11.5, loc="left", pad=12)
ax.set_xlabel("Precision@50 within a held-out fold", color=INK_MUTED, fontsize=9.5)
ax.set_xlim(0.45, 1.0)
fig.tight_layout()
fig.savefig(fig_dir / "fig3_fold_spread.svg", format="svg", facecolor=SURFACE)
plt.close(fig)

print("wrote work/figures/fig1_label_rate_by_score.svg")
print("wrote work/figures/fig2_precision_at_k.svg")
print("wrote work/figures/fig3_fold_spread.svg")
print()

# The table views, so the figures are never the only way to read the numbers.
print("Table view of Figure 1:")
print(by_score.rename(columns={"size": "pages", "mean": "declining_rate"}).round(3).to_string())
print()
print("Table view of Figure 2:")
print(pd.DataFrame({"K": ks, "ML07_rule": rule_ps, "hybrid_shipped": hyb_ps,
                    "base_rate": BASE_RATE}).round(3).to_string(index=False))
print()
print("Table view of Figure 3 (precision@50 per fold):")
f3 = pd.DataFrame({LABELS[n]: folds_pf[n] for n in ORDER}).T
f3.columns = [f"fold{i+1}" for i in range(f3.shape[1])]
f3["mean"] = f3.mean(axis=1).round(3)
f3["worst"] = f3.iloc[:, :5].min(axis=1)
print(f3.round(3).to_string())

wrote work/figures/fig1_label_rate_by_score.svg
wrote work/figures/fig2_precision_at_k.svg
wrote work/figures/fig3_fold_spread.svg

Table view of Figure 1:
                pages  declining_rate
baseline_score                       
0                  75           0.293
1                1809           0.088
2                 612           0.350
3                2785           0.417
4                3356           0.449
5                4331           0.577
6                5145           0.593
7                2606           0.467
8                6291           0.684
9                2990           0.714

Table view of Figure 2:
  K  ML07_rule  hybrid_shipped  base_rate
 10      0.700           1.000      0.542
 20      0.750           0.950      0.542
 50      0.740           0.900      0.542
100      0.800           0.850      0.542
200      0.835           0.805      0.542
500      0.808           0.770      0.542

Table view of Figure 3 (precision@50 per fold):
                    

## ML-12 — Demo, social cut, and employer summary

### 5-minute demo outline

| Time | Beat | What is on screen |
|---|---|---|
| 0:00–0:45 | **The decision, not the model.** An editor has 50 slots and 30,000 pages. Today triage is "anything not touched in six months" — and in this data that rule fires on **17 pages**. | The freshness-tier table: 174 pages at 181+ days |
| 0:45–1:15 | **The metric, chosen before building.** Precision@50 against a 54.2% base rate. Flag-everything already scores 0.542, so any number without its base rate is not a claim. | Figure 2, base-rate line only |
| 1:15–2:00 | **The leakage find.** The label is a −20% threshold on the 30-day impression pair — so those two columns reconstruct it **exactly** (1.0000). The docs name two forbidden fields; the real list is four. | The reconstruction printout: 1.0000 / 0.5364 / 0.5383 |
| 2:00–2:45 | **The baseline, and the defect I found in it.** Five readable conditions: 0.740 precision@50 vs 0.542. Then precision *rises* with K, and my three top-ranked pages had all grown. Good band, bad ordering. | Figure 1, then Figure 2 with the rule curve |
| 2:45–3:30 | **The split that decides whether any of this is real.** Random row split: AUC 0.777. Grouped by client: 0.624. **That 0.15 is client memorisation** — the easiest way to publish a wrong number on this data. | The two-split comparison printout |
| 3:30–4:15 | **The model, and the prediction it was built to test.** Hybrid: rule band + logistic ordering. **0.900 precision@50 — 45 of 50 slots** — and the only curve that falls monotonically from 1.000 at K=10. The prediction was recorded before training; three candidates failed it. | Figure 2, both curves |
| 4:15–5:00 | **What it still cannot say, including where I would overrule myself.** It ranks by *exposure*, not decay — its top feature by 2.4×. Every top-50 miss is a page-1 page that grew. Boosting holds a better floor across unseen clients than the system I shipped. And the queue now spends 58% of a sprint on one client. | Figure 3 + limitations slide |

### Social-post cut

> Spent a week ranking 30,000 pages for editorial refresh. The most useful findings were all in my own
> numbers.
>
> **1.** A transparent 5-condition rule hit **0.740 precision@50** against a **0.542** base rate. Then
> precision *rose* with K — 0.740 at 50, 0.835 at 200. That only happens if the top of your ranking is
> wrong. The three pages it ranked highest had all **grown** 35–69%.
>
> **2.** The fix: keep the rule for the band and the reason codes, let a model order inside it.
> **0.900 precision@50 — 45 of 50 slots on genuinely declining pages** — and the only curve of the four I
> tested that falls monotonically from 1.000. The readable model beat both tree ensembles, but *only*
> after log-transforming the skewed count columns. Feature engineering outranked model choice.
>
> **3.** The number I nearly published instead: a random train/test split scored **AUC 0.777**. Grouping by
> client dropped it to **0.624**. That 0.15 was the model memorising which client a page belonged to.
>
> **4.** What it still cannot do: its top feature is impressions volume by 2.4×, so it ranks by *exposure*,
> not decay — and every miss in its top 50 is a page-1 page that was actually growing.
>
> Built on the FlyRank ML Internship dataset. #MachineLearning #SEO #DataScience

### Employer-facing summary (3 sentences)

> I built a ranked editorial-review queue over 30,000 pseudonymized content items, framing it as a
> capacity-constrained ranking problem and committing to precision@50 against the 54.2% base rate before
> building anything. A transparent five-condition rule reached 0.740; I then diagnosed its specific defect
> — precision *rising* with K, meaning the top of the ranking was mis-ordered — recorded that as a
> falsifiable test, and shipped a hybrid that keeps the rule's band and reason codes while a logistic model
> orders within it, reaching **0.900 precision@50 on a client-held-out split** (45 of 50 slots vs 27 for
> random triage) as the only one of four candidates whose precision curve falls monotonically. The work is
> reproducible from a fresh clone with committed metrics receipts, and every result that runs against my
> own conclusion is reported alongside it — the 0.15 AUC of client memorisation a random split would have
> handed me, the volume-not-decay feature profile I flagged as a risk before training and then reproduced,
> and the fact that a model I did not ship holds a better floor across unseen clients.

## Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset** — [flyrank.ai](https://flyrank.ai). Thanks to the FlyRank
team for the pseudonymized data release and the lane framing.

*(This section is required at the bottom of the deployed paper, alongside the Abstract at the top — see
`work/capstone_report.md` §0 and §9.)*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Every model number is reproduced by this notebook and asserted against the committed ML-08 receipt
- [x] The results that argue against my own choice are in the paper, not omitted from it
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.